In [1]:
import os 
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import torch
import csv, random
import time
import pandas as pd
import itertools
from pathlib import Path

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


In [3]:
def load_png_as_rgb(png_path):
    with Image.open(png_path) as img:
        img = img.convert("RGB")
        arr = np.array(img, dtype = np.uint8)

    assert arr.ndim == 3 and arr.shape[2] == 3
    assert arr.dtype == np.uint8
    return arr

def show_anns(anns, borders=True):
    if len(anns) == 0:
        return
    sorted_anns = sorted(anns, key=(lambda x: x['area']), reverse=True)
    ax = plt.gca()
    ax.set_autoscale_on(False)

    img = np.ones((sorted_anns[0]['segmentation'].shape[0], sorted_anns[0]['segmentation'].shape[1], 4))
    img[:, :, 3] = 0
    for ann in sorted_anns:
        m = ann['segmentation']
        color_mask = np.concatenate([np.random.random(3), [0.5]])
        img[m] = color_mask 
        if borders:
            import cv2
            contours, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE) 
            # Try to smooth contours
            contours = [cv2.approxPolyDP(contour, epsilon=0.01, closed=True) for contour in contours]
            cv2.drawContours(img, contours, -1, (0, 0, 1, 0.4), thickness=1) 

    ax.imshow(img)

def iou_binary(pred_bool, gt_bool):
    inter = np.logical_and(pred_bool, gt_bool).sum()
    uni   = np.logical_or(pred_bool, gt_bool).sum()
    return inter / uni if uni > 0 else float("nan")

In [4]:
from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator

In [5]:
sam2_checkpoint = str(Path(os.environ.get("SAM2_CHECKPOINT", "../checkpoints/sam2.1_hiera_large.pt")))
model_cfg = str(Path("configs/sam2.1/sam2.1_hiera_l.yaml"))

In [6]:
sam2 = build_sam2(model_cfg, sam2_checkpoint, device=device, apply_postprocessing=False)

In [7]:
paired_csv = Path.home() / "skimage_segmentation" / "paired.csv"

with paired_csv.open(newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

print("rows:", len(rows))

rows: 135


In [8]:
def compute_mask_metrics(masks, H, W):
    num_masks = len(masks)
    if num_masks == 0:
        return {
            "num_masks": 0,
            "union_coverage": 0.0,
            "overlap_2plus": 0.0,
            "median_area": 0.0,
            "p90_area": 0.0,
            "pred_iou_mean": np.nan,
            "pred_iou_min": np.nan,
            "stability_mean": np.nan,
            "stability_min": np.nan,
        }

    union = np.zeros((H, W), dtype=bool)
    cover_count = np.zeros((H, W), dtype=np.uint16)
    areas = []
    ious = []
    stabs = []

    for ann in masks:
        m = ann["segmentation"].astype(bool)
        s = int(m.sum())
        if s == 0:
            continue

        union |= m
        cover_count[m] += 1
        areas.append(s)

        if "predicted_iou" in ann:
            ious.append(float(ann["predicted_iou"]))
        if "stability_score" in ann:
            stabs.append(float(ann["stability_score"]))

    union_coverage = float(union.mean())
    overlap_2plus = float((cover_count >= 2).mean())

    if len(areas) == 0:
        median_area = 0.0
        p90_area = 0.0
    else:
        areas = np.array(areas, dtype=np.int64)
        median_area = float(np.median(areas))
        p90_area = float(np.quantile(areas, 0.90))

    ious = np.array(ious, dtype=float) if len(ious) else np.array([np.nan])
    stabs = np.array(stabs, dtype=float) if len(stabs) else np.array([np.nan])

    return {
        "num_masks": int(num_masks),
        "union_coverage": union_coverage,
        "overlap_2plus": overlap_2plus,
        "median_area": median_area,
        "p90_area": p90_area,
        "pred_iou_mean": float(np.nanmean(ious)),
        "pred_iou_min": float(np.nanmin(ious)),
        "stability_mean": float(np.nanmean(stabs)),
        "stability_min": float(np.nanmin(stabs)),
    }

def run_e2e_labeling(masks, ice_mask_bool):
     H, W = ice_mask_bool.shape
     sam2_pred = np.full((H, W), -1, dtype=np.int8)

     masks_sorted = sorted(masks, key=lambda x: x.get("predicted_iou", 0.0), reverse=True)
     water_mask = ice_mask_bool.astype(bool)

     for ann in masks_sorted:
        m = ann["segmentation"].astype(bool)
        if m.sum() == 0:
            continue

        mm = m & (sam2_pred == -1)
        if mm.sum() == 0:
            continue

        water_fraction = water_mask[mm].mean()
        sam2_pred[mm] = 1 if water_fraction > 0.5 else 0

     return sam2_pred

In [12]:
best_cfg = {
    "points_per_side": 48,
    "points_per_batch": 128,
    "pred_iou_thresh": 0.3,
    "stability_score_thresh": 0.8,
    "stability_score_offset": 0.6,
    "crop_n_layers": 0,
    "box_nms_thresh": 0.8,
    "crop_n_points_downscale_factor": 2,
    "min_mask_region_area": 20.0,
    "use_m2m": True,
}

def water_fraction_from_npz(npz_path):
    d = np.load(npz_path)
    m = d["ice_mask"].astype(bool)
    return float(m.mean())

good_idxs = []
for idx in range(len(rows)):
    npz_path = rows[idx]["npz_path"].strip()
    try:
        f = water_fraction_from_npz(npz_path)
    except Exception:
        continue
    if 0.05 < f < 0.95:
        good_idxs.append(idx)

print("good examples:", len(good_idxs))

pool = good_idxs if len(good_idxs) >= 10 else list(range(len(rows)))
#test_idxs = sorted(random.sample(pool, k=min(10, len(pool))))
test_idxs = [11, 20, 27, 30, 56, 74, 83, 119, 128, 129]
print("test idxs:", test_idxs)

good examples: 25
test idxs: [11, 20, 27, 30, 56, 74, 83, 119, 128, 129]


In [13]:
gen = SAM2AutomaticMaskGenerator(model=sam2, **best_cfg)

results = []
vizual_cache = []  

for n, idx in enumerate(test_idxs, 1):
    png_path = rows[idx]["png_path"].strip()
    npz_path = rows[idx]["npz_path"].strip()

    image = load_png_as_rgb(png_path)
    d = np.load(npz_path)
    ice_mask = d["ice_mask"].astype(bool)

    H, W = image.shape[:2]

    t0 = time.perf_counter()
    masks = gen.generate(image)
    t_s = time.perf_counter() - t0

    row = {
        "idx": idx,
        "time_s": float(t_s),
        "water_frac_gt": float(ice_mask.mean()),
        "png_path": png_path,
        "npz_path": npz_path,
    }
    row.update(compute_mask_metrics(masks, H, W))

    sam2_pred = run_e2e_labeling(masks, ice_mask)
    valid = (sam2_pred != -1)
    row["covered_frac"] = float(valid.mean())
    row["e2e_iou_water_cov"] = float(iou_binary((sam2_pred == 1) & valid, (ice_mask == True) & valid))
    row["e2e_iou_ice_cov"]   = float(iou_binary((sam2_pred == 0) & valid, (ice_mask == False) & valid))
    row["e2e_iou_water_full"] = float(iou_binary((sam2_pred == 1), (ice_mask == True)))
    row["e2e_iou_ice_full"]   = float(iou_binary((sam2_pred == 0), (ice_mask == False)))

    results.append(row)
    
    if n <= 2:
        vizual_cache.append((idx, image, masks, ice_mask, sam2_pred))

    print(f"done {n}/{len(test_idxs)} | idx={idx} | time={t_s:.3f}s | masks={len(masks)} | covered={row['covered_frac']:.3f}")

df_test = pd.DataFrame(results)

/home/predm/sam2/sam2/sam2_image_predictor.py:431: UserWarning: cannot import name '_C' from 'sam2' (/home/predm/sam2/sam2/__init__.py)

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although some post-processing functionality may be limited (which doesn't affect the results in most cases; see https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).
  masks = self._transforms.postprocess_masks(


done 1/10 | idx=11 | time=12.761s | masks=76 | covered=0.989
done 2/10 | idx=20 | time=21.290s | masks=62 | covered=0.989
done 3/10 | idx=27 | time=11.847s | masks=104 | covered=0.992
done 4/10 | idx=30 | time=21.449s | masks=28 | covered=0.989
done 5/10 | idx=56 | time=21.092s | masks=47 | covered=0.982
done 6/10 | idx=74 | time=11.422s | masks=91 | covered=0.994
done 7/10 | idx=83 | time=12.004s | masks=115 | covered=0.267
done 8/10 | idx=119 | time=12.166s | masks=175 | covered=0.419
done 9/10 | idx=128 | time=11.755s | masks=124 | covered=0.277
done 10/10 | idx=129 | time=12.211s | masks=170 | covered=0.482


In [14]:
cols_main = [
    "idx", 
    "time_s", 
    "num_masks", 
    "covered_frac", 
    "union_coverage", 
    "overlap_2plus"
]

display(df_test[cols_main].sort_values("covered_frac", ascending=False))

print("\nSummary (mean ± std):")
for col in ["time_s", "num_masks", "covered_frac", "union_coverage", "overlap_2plus"]:
    vals = df_test[col].astype(float)
    print(f"{col:>16}: {vals.mean():.4f} ± {vals.std():.4f}")

,idx,time_s,num_masks,covered_frac,union_coverage,overlap_2plus
5,74,11.421761,91,0.993538,0.993538,0.178699
2,27,11.846630,104,0.992252,0.992252,0.100594
3,30,21.449260,28,0.989330,0.989330,0.029701
0,11,12.760869,76,0.988907,0.988907,0.206871
1,20,21.290385,62,0.988705,0.988705,0.099781
4,56,21.092070,47,0.982101,0.982101,0.190304
9,129,12.211283,170,0.482140,0.482140,0.353878
7,119,12.166247,175,0.419338,0.419338,0.134850
8,128,11.754677,124,0.276943,0.276943,0.131413
6,83,12.003786,115,0.267090,0.267090,0.220589



Summary (mean ± std):
          time_s: 14.7997 ± 4.4839
       num_masks: 99.2000 ± 48.7962
    covered_frac: 0.7380 ± 0.3300
  union_coverage: 0.7380 ± 0.3300
   overlap_2plus: 0.1647 ± 0.0882


In [ ]:
df_test.to_csv("test.csv", index=False)